Plan: Semantic Annotation Evaluation:
1. Load Saved Annotations — Load the 61 JSON files without making additional Gemini API requests.
2. Shape Analysis — Analyze the distribution of round, oval, irregular, and uncertain shapes.
3. Appearance Analysis — Examine textual characteristics such as anechoic, hypoechoic, and well-defined borders.
4. Image Quality Analysis — Analyze the distribution of image quality ratings: good, moderate, and poor.
5. Uncertainty Analysis — Determine how many images Gemini flagged as uncertain.
6. Qualitative Evaluation — Select 15–20 images for manual assessment of whether the generated descriptions are supported by visible image features.
7. Save Results — Save the evaluation tables and visualizations for the Bachelor thesis.

In [ ]:
from google.colab import drive
from pathlib import Path
import pandas as pd
import json

drive.mount("/content/drive")

PROJECT_DIR = Path(
    "/content/drive/MyDrive/ovarian_ultrasound_vlm"
)

MANIFEST_PATH = PROJECT_DIR / "image_manifest.csv"

PREDICTIONS_DIR = (
    PROJECT_DIR / "predictions" / "main_validation"
)

manifest = pd.read_csv(MANIFEST_PATH)

validation = (
    manifest[manifest["split"] == "validation"]
    .sort_values("case_id")
    .reset_index(drop=True)
)

assert len(validation) == 61

# Load existing predictions without calling Gemini API.
annotations = []

for _, row in validation.iterrows():
    case_id = row["case_id"]
    prediction_path = PREDICTIONS_DIR / f"{case_id}.json"

    assert prediction_path.exists(), (
        f"Missing annotation: {case_id}"
    )

    with open(prediction_path, "r", encoding="utf-8") as f:
        result = json.load(f)

    ann = result["annotation"]

    annotations.append({
        "case_id": case_id,
        "category": row["category"],
        "model": result["model"],
        "follicle_like_count": ann["follicle_like_count"],
        "image_quality": ann["image_quality"],
        "uncertain": ann["uncertain"],
        "description": ann["description"],
        "regions": ann["regions"]
    })

semantic_df = pd.DataFrame(annotations)

assert len(semantic_df) == 61
assert semantic_df["case_id"].is_unique

print("Loaded annotations:", len(semantic_df))
print(
    "Total described regions:",
    semantic_df["follicle_like_count"].sum()
)

display(
    semantic_df[
        [
            "case_id",
            "category",
            "follicle_like_count",
            "image_quality",
            "uncertain",
            "description"
        ]
    ].head(10)
)

 Load the saved Gemini validation annotations for semantic evaluation

In [ ]:
# Create a region-level DataFrame from Gemini annotations.

region_records = []

for _, row in semantic_df.iterrows():
    for region_id, region in enumerate(row["regions"], start=1):
        region_records.append({
            "case_id": row["case_id"],
            "category": row["category"],
            "region_id": region_id,
            "shape": region["shape"],
            "appearance": region["appearance"],
            "bbox": region["bbox"]
        })

regions_df = pd.DataFrame(region_records)

assert len(regions_df) == 143

print("Total annotated regions:", len(regions_df))

# Analyze the distribution of predicted shapes.
shape_counts = (
    regions_df["shape"]
    .value_counts()
    .reindex(
        ["round", "oval", "irregular", "uncertain"],
        fill_value=0
    )
)

shape_distribution = pd.DataFrame({
    "count": shape_counts,
    "percentage": (
        shape_counts / len(regions_df) * 100
    ).round(2)
})

print("\nPredicted shape distribution:")
display(shape_distribution)

print("\nShape distribution by category:")
display(
    pd.crosstab(
        regions_df["category"],
        regions_df["shape"]
    )
)

Extract region-level annotations and analyze predicted shape distributions

In [ ]:
import matplotlib.pyplot as plt

FIGURES_DIR = PROJECT_DIR / "figures" / "semantic_evaluation"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

fig, ax = plt.subplots(figsize=(8, 5))

shape_counts.plot(
    kind="bar",
    ax=ax,
    color="steelblue"
)

ax.set_title("Gemini-Predicted Region Shapes")
ax.set_xlabel("Predicted shape")
ax.set_ylabel("Number of regions")
ax.tick_params(axis="x", rotation=0)

plt.tight_layout()

figure_path = FIGURES_DIR / "shape_distribution.png"

fig.savefig(
    figure_path,
    dpi=200,
    bbox_inches="tight"
)

plt.show()

print("Figure saved:", figure_path)

Visualize and save the distribution of Gemini-predicted region shapes

In [ ]:
import re
import pandas as pd

# Define keyword patterns for appearance analysis.
appearance_patterns = {
    "anechoic": r"\banechoic\b",
    "hypoechoic": r"\bhypoechoic\b",
    "well_defined": (
        r"\bwell[- ]defined\b|"
        r"\bwell[- ]circumscribed\b"
    ),
    "smooth_borders": r"\bsmooth\b",
    "irregular_borders": r"\birregular\b",
    "cystic": r"\bcystic\b|\bcyst[- ]like\b"
}

# Create a copy to preserve the original annotations.
appearance_df = regions_df.copy()

for characteristic, pattern in appearance_patterns.items():
    appearance_df[characteristic] = (
        appearance_df["appearance"]
        .fillna("")
        .str.contains(
            pattern,
            flags=re.IGNORECASE,
            regex=True
        )
    )

# Calculate the frequency of each characteristic.
appearance_counts = appearance_df[
    list(appearance_patterns.keys())
].sum().sort_values(ascending=False)

appearance_distribution = pd.DataFrame({
    "count": appearance_counts,
    "percentage": (
        appearance_counts / len(appearance_df) * 100
    ).round(2)
})

print("Total region descriptions:", len(appearance_df))
print("\nAppearance characteristic distribution:")

display(appearance_distribution)

Extract visual appearance characteristics from Gemini-generated region descriptions

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(9, 5))

appearance_counts.plot(
    kind="barh",
    ax=ax,
    color="steelblue"
)

ax.set_title("Gemini-Predicted Appearance Characteristics")
ax.set_xlabel("Number of region descriptions")
ax.set_ylabel("Appearance characteristic")

ax.invert_yaxis()

plt.tight_layout()

figure_path = (
    FIGURES_DIR / "appearance_distribution.png"
)

fig.savefig(
    figure_path,
    dpi=200,
    bbox_inches="tight"
)

plt.show()

print("Figure saved:", figure_path)

Visualize and save the distribution of predicted appearance characteristics

In [ ]:
# Calculate image quality distribution.
quality_counts = (
    semantic_df["image_quality"]
    .value_counts()
    .reindex(["good", "moderate", "poor"], fill_value=0)
)

quality_distribution = pd.DataFrame({
    "count": quality_counts,
    "percentage": (
        quality_counts / len(semantic_df) * 100
    ).round(2)
})

print("Image quality distribution:")
display(quality_distribution)

print("\nImage quality by category:")
display(
    pd.crosstab(
        semantic_df["category"],
        semantic_df["image_quality"]
    )
)

Analyze Gemini-predicted image quality across the validation dataset

In [ ]:
# Count uncertain and non-uncertain image annotations.
uncertainty_counts = (
    semantic_df["uncertain"]
    .value_counts()
    .reindex([False, True], fill_value=0)
)

print("Uncertainty distribution:")
display(pd.DataFrame({
    "count": uncertainty_counts,
    "percentage": (
        uncertainty_counts / len(semantic_df) * 100
    ).round(2)
}))

print("\nUncertainty by category:")
display(
    pd.crosstab(
        semantic_df["category"],
        semantic_df["uncertain"]
    )
)

print("\nRegion counts by uncertainty:")
display(
    semantic_df.groupby("uncertain")["follicle_like_count"]
    .agg(["count", "mean", "median", "min", "max"])
)   

Analyze Gemini-reported uncertainty and its relationship to region counts

In [ ]:
import numpy as np
import pandas as pd

SEED = 42

# Stratified sample from the original dataset categories
sample_sizes = {
    "dominant": 10,
    "normal": 3,
    "pco": 7
}

selected_ids = []

for category, n in sample_sizes.items():
    category_df = semantic_df[
        semantic_df["category"] == category
    ]

    sampled = category_df.sample(
        n=n,
        random_state=SEED
    )

    selected_ids.extend(sampled["case_id"].tolist())

qualitative_sample = (
    semantic_df[
        semantic_df["case_id"].isin(selected_ids)
    ]
    .sort_values("case_id")
    .reset_index(drop=True)
)

assert len(qualitative_sample) == 20
assert qualitative_sample["case_id"].is_unique

print("Qualitative sample size:", len(qualitative_sample))

print("\nCategory distribution:")
print(qualitative_sample["category"].value_counts())

print("\nUncertainty distribution:")
print(qualitative_sample["uncertain"].value_counts())

display(
    qualitative_sample[
        [
            "case_id",
            "category",
            "follicle_like_count",
            "image_quality",
            "uncertain"
        ]
    ]
)

Select a reproducible, category-stratified subset for qualitative annotation review

In [ ]:
SEMANTIC_DIR = (
    PROJECT_DIR / "evaluation" / "semantic"
)

SEMANTIC_DIR.mkdir(parents=True, exist_ok=True)

sample_path = (
    SEMANTIC_DIR / "qualitative_sample.csv"
)

qualitative_sample[
    [
        "case_id",
        "category",
        "follicle_like_count",
        "image_quality",
        "uncertain"
    ]
].to_csv(sample_path, index=False)

print("Qualitative sample saved:", sample_path)

Save the selected cases for reproducible qualitative evaluation

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
import textwrap
from PIL import Image

REVIEW_DIR = (
    PROJECT_DIR / "figures" / "semantic_evaluation" / "qualitative_review"
)
REVIEW_DIR.mkdir(parents=True, exist_ok=True)

manifest_lookup = manifest.set_index("case_id")

for _, row in qualitative_sample.iterrows():
    case_id = row["case_id"]
    metadata = manifest_lookup.loc[case_id]

    image = np.array(
        Image.open(metadata["image_path"]).convert("RGB")
    )

    mask = np.array(
        Image.open(metadata["pseudo_label_path"])
    )

    height, width = image.shape[:2]

    fig, axes = plt.subplots(1, 2, figsize=(12, 6))

    # Original image
    axes[0].imshow(image)
    axes[0].set_title("Original Ultrasound")
    axes[0].axis("off")

    # Image with BiomedParse mask and Gemini boxes
    axes[1].imshow(image)

    overlay = np.zeros((height, width, 4), dtype=float)
    overlay[mask == 1, :3] = [0.0, 0.5, 1.0]
    overlay[mask == 1, 3] = 0.45

    axes[1].imshow(overlay)

    for region in row["regions"]:
        x1, y1, x2, y2 = region["bbox"]

        xmin = x1 / 1000 * width
        ymin = y1 / 1000 * height
        xmax = x2 / 1000 * width
        ymax = y2 / 1000 * height

        rect = patches.Rectangle(
            (xmin, ymin),
            xmax - xmin,
            ymax - ymin,
            linewidth=2,
            edgecolor="lime",
            facecolor="none"
        )
        axes[1].add_patch(rect)

    axes[1].set_title(
        "BiomedParse Pseudo-label + Gemini Boxes"
    )
    axes[1].axis("off")

    # Add Gemini image-level description.
    wrapped_description = textwrap.fill(
        str(row["description"]),
        width=90
    )

    fig.suptitle(
        f"{case_id} | Gemini count: {row['follicle_like_count']} "
        f"| Uncertain: {row['uncertain']}",
        fontsize=13
    )

    fig.text(
        0.5, 0.04,
        wrapped_description,
        ha="center",
        va="bottom",
        fontsize=9
    )

    plt.tight_layout(rect=[0, 0.12, 1, 0.92])

    output_path = REVIEW_DIR / f"{case_id}_review.png"

    fig.savefig(
        output_path,
        dpi=180,
        bbox_inches="tight"
    )

    plt.close(fig)

print("Qualitative review figures saved:", REVIEW_DIR)
print("Number of figures:", len(list(REVIEW_DIR.glob("*_review.png"))))

Generate individual review figures with ultrasound images, pseudo-labels, Gemini boxes, and descriptions